# 5. Grey numbers

A *grey number* ⊗ = `[lower, upper]` is a quantity known only to lie within an interval. turboswarm can optimize over grey variables: the swarm searches each one's **center** and **spread** (half-width), the whole interval is kept inside the bounds you set, and your objective collapses a grey result to a single value (whitenization). Run `maturin develop --release` first.

In [ ]:
import matplotlib.pyplot as plt
import turboswarm as pso

## A native grey benchmark

`grey_sphere` rewards both accuracy (centers at 0) and certainty (zero spread); its optimum is the crisp ⊗ = `[0, 0]`. Passing the name runs it in Rust (no GIL). `grey_benchmark_info` returns its recommended `(center_bound, max_spread, optimum)`.

In [ ]:
cb, ms, opt = pso.grey_benchmark_info("grey_sphere")
r = pso.minimize_grey("grey_sphere", bounds=(-cb, cb), dim=2, seed=42)
print("best value    :", r.best_value)
print("best intervals:", [(round(lo, 4), round(hi, 4)) for (lo, hi) in r.best_position])

## Your own grey objective

The objective receives a list of `(lower, upper)` intervals and returns a `float`. Here we drive each center to 2 while lightly penalizing uncertainty. The result exposes the solution as intervals (`best_position`) and as centers/spreads.

In [ ]:
def grey_quadratic(greys):
    centers = [(lo + hi) / 2 for (lo, hi) in greys]
    spreads = [(hi - lo) / 2 for (lo, hi) in greys]
    return sum((c - 2) ** 2 for c in centers) + 0.5 * sum(spreads)


r = pso.minimize_grey(grey_quadratic, bounds=(-10, 10), dim=3, seed=1)
print("centers:", [round(c, 3) for c in r.best_centers])
print("spreads:", [round(s, 3) for s in r.best_spreads])

## Choosing the representation

With `representation="center_spread"` the objective receives `(center, spread)` pairs instead of `(lower, upper)` — often more natural. The math and the seed are the same, so the result is identical; only the input form changes.

In [ ]:
def grey_quadratic_cs(greys):
    return sum((c - 2) ** 2 for (c, s) in greys) + 0.5 * sum(s for (c, s) in greys)


r_cs = pso.minimize_grey(
    grey_quadratic_cs, bounds=(-10, 10), dim=3, seed=1, representation="center_spread"
)
print("same result as the interval form:", r_cs.best_value == r.best_value)

## Bounding each grey number

`bounds` are the `(lower, upper)` limits the *whole* interval must stay within. To see it, we maximize the total spread: each grey number widens until it fills its limits. `max_spread` adds an optional extra cap on the half-width.

In [ ]:
def maximize_spread(greys):
    return -sum(s for (c, s) in greys)


# Per-variable limits: the intervals widen to exactly [-2, 3] and [0, 1].
r = pso.minimize_grey(
    maximize_spread, bounds=[(-2, 3), (0, 1)], representation="center_spread", seed=11
)
print("widest intervals:", [(round(lo, 3), round(hi, 3)) for (lo, hi) in r.best_position])

# An extra max_spread cap limits the half-width even inside wide bounds.
r2 = pso.minimize_grey(
    maximize_spread, bounds=(-10, 10), dim=2, max_spread=1.5,
    representation="center_spread", seed=3,
)
print("spreads capped at 1.5:", [round(s, 3) for s in r2.best_spreads])

## The convergence curve

`GreyResult` carries the same `convergence` trace as a regular run, so `viz.plot_convergence` works unchanged.

In [ ]:
r = pso.minimize_grey("grey_sphere", bounds=(-5.12, 5.12), dim=2, seed=42)
pso.viz.plot_convergence(r)
plt.show()